In [1]:
import math
import time
import random
import itertools as it
from collections import defaultdict

In [2]:
MODES = [True, False]
# MODES = [True]

In [3]:
def get_uni_rr_old(rr: str, max: bool = True) -> str:
    """
    Returns the unique canonical representation (maximal truth table or 
    minimal Blake canonical form) of a Boolean rule.
    
    This function processes a binary string of length 2^k (where k is the 
    number of regulators) by identifying and filling out sub-cubes of a 
    hypercube using fast bitwise integer logic.

    Parameters
    ----------
    rr  - representation of the rule            : length 2^k binary str
    max - if True, get maximal representation   : bool
          if False, get minimal representation

    Returns
    -------
    uni_rr - unique representation of the rule  : length 2^k binary str
    """
    n = len(rr)
    
    # Base case: if the string length is 1 or less (k = 0 regulators),
    # the rule represents a fixed node configuration and cannot be minimized further.
    if n <= 1:
        return rr

    # Convert the string to a list of characters to allow fast, 
    # in-place mutations at specific index positions.
    uni_rr = list(rr)
    
    # Store already visited sub-cube positions in a set to ensure 
    # instant O(1) lookup speeds and prevent redundant evaluations.
    modified = set()
    fill_value = '1' if max else '0'

    # Iterate through all binary rule configurations in reverse order (from n-1 down to 0).
    # Processing in reverse ensures that higher-order implicants are evaluated first.
    for i in range(n):
        # Calculate the actual array index corresponding to the reversed loop variable.
        rev_idx = n - i - 1
        
        # We only evaluate active rule states ('1'). Inactive states ('0') 
        # do not trigger prime implicant/sub-cube expansions.
        if uni_rr[rev_idx] != '1':
            continue
            
        # Skip this position if it has already been covered and filled 
        # by a previously processed sub-cube.
        if i in modified:
            continue

        # Scan the entire state space to locate sub-cube coordinates.
        # Instead of slow string generation, we use low-level bitwise operations.
        for position in range(n):
            if position == i:
                continue
                
            # BITWISE LOGIC: (position & i) == i
            # Checks if 'position' contains a 1 at every single binary slot where 'i' has a 1.
            # If True, 'position' is mathematically verified to be a sub-cube coordinate
            # covered by the root implicant 'i'.
            if (position & i) == i:
                modified.add(position)
                
                # Maps the integer position back to its correct index in the output array.
                target_idx = n - position - 1
                uni_rr[target_idx] = fill_value

    # Recombine the character array into the final canonical binary string representation.
    return ''.join(uni_rr)

In [4]:
def get_uni_rr(rr: str, max: bool = True) -> str:
    rr = rr[::-1]
    n = len(rr)
    
    if n <= 1:
        return rr

    uni_rr = list(rr)
    
    modified = set()
    fill_value = '1' if max else '0'

    for i in range(n):
        if uni_rr[i] != '1':
            continue
            
        if i in modified:
            continue

        for position in range(i+1, n):               
            if (position & i) == i:
                modified.add(position)
                
                uni_rr[position] = fill_value

    return ''.join(uni_rr)[::-1]

In [5]:
def benchmark_and_verify():
    # Generate test strings of lengths 4 (k=2) and 8 (k=3) exhaustively
    # Plus longer length 16 (k=4) strings to show scaling performance differences
    test_cases = []
    
    # 1. Exhaustive combos for length 4 and 8
    for length in [4, 8, 16]:
        test_cases.extend(["".join(seq) for seq in it.product(['0', '1'], repeat=length)])
        
    # 2. Add 20 random 2048-length (k=11) strings using a fixed seed for consistency
    random.seed(42)  # Set seed to ensure the exact same strings generate every time
    
    for _ in range(20):
        # Generate a random list of '0's and '1's of length 2048
        random_bits = [random.choice(['0', '1']) for _ in range(2048)]
        test_cases.append("".join(random_bits))

    print(f"📊 Running differential testing on {len(test_cases)} test cases...")
    print(f"🧬 (Includes 20 randomized rule strings of length 2048)")

    print(f"📊 Running differential testing on {len(test_cases)} test cases...")

    # --- Benchmark the OLD Function ---
    start_old = time.perf_counter()
    old_results = []
    for s in test_cases:
        for mode in MODES:
            old_results.append(get_uni_rr_old(s, max=mode))
    end_old = time.perf_counter()
    time_old = end_old - start_old

    # --- Benchmark the NEW Function ---
    start_new = time.perf_counter()
    new_results = []
    for s in test_cases:
        for mode in MODES:
            new_results.append(get_uni_rr(s, max=mode))
    end_new = time.perf_counter()
    time_new = end_new - start_new

    # --- Output Verification Check ---
    if old_results != new_results:
        print("❌ CRITICAL ERROR: The outputs do NOT match! Look for logical differences.")
        # Find where it failed
        idx = 0
        for s in test_cases:
            for mode in MODES:
                if old_results[idx] != new_results[idx]:
                    print(f"Mismatch found at input: {s} (max={mode})")
                    print(f"Old: {old_results[idx]}")
                    print(f"New: {new_results[idx]}")
                    return
                idx += 1
        return

    # --- Print Benchmark Report ---
    print("\n" + "="*40)
    print("      🏎️  BENCHMARK RESULTS 🏎️      ")
    print("="*40)
    print(f"✅ Output Status:      MATCHED EXACTLY")
    print(f"⏱️  Old Algorithm:     {time_old:.6f} seconds")
    print(f"⚡ New Algorithm:     {time_new:.6f} seconds")
    
    speedup = time_old / time_new if time_new > 0 else float('inf')
    print(f"🚀 Speed Multiplier:   {speedup:.2f}x FASTER")
    print("="*40)

# Run the benchmark tool
benchmark_and_verify()

📊 Running differential testing on 65828 test cases...
🧬 (Includes 20 randomized rule strings of length 2048)
📊 Running differential testing on 65828 test cases...

      🏎️  BENCHMARK RESULTS 🏎️      
✅ Output Status:      MATCHED EXACTLY
⏱️  Old Algorithm:     1.172329 seconds
⚡ New Algorithm:     0.632628 seconds
🚀 Speed Multiplier:   1.85x FASTER


In [6]:
def benchmark_by_size_with_seed():
    # Group test cases by string length
    cases_by_size = defaultdict(list)
    
    # 1. Exhaustive small combinations (lengths 4, 8, 16)
    for length in [4, 8, 16]:
        combos = ["".join(seq) for seq in it.product(['0', '1'], repeat=length)]
        cases_by_size[length].extend(combos)
        
    # 2. Add 20 random 2048-length strings using a fixed seed
    random.seed(42)
    for _ in range(20):
        random_bits = [random.choice(['0', '1']) for _ in range(2048)]
        cases_by_size[2048].append("".join(random_bits))

    print("📊 Running size-isolated differential testing...")
    print("=" * 60)
    print(f"{'String Size':<12} | {'Old Time (s)':<14} | {'New Time (s)':<14} | {'Speedup Factor':<14}")
    print("-" * 60)

    # Process each size category independently
    for size, strings in sorted(cases_by_size.items()):
        
        # --- Time the OLD Function ---
        start_old = time.perf_counter()
        old_results = []
        for s in strings:
            for mode in MODES:
                old_results.append(get_uni_rr_old(s, max=mode))
        end_old = time.perf_counter()
        time_old = end_old - start_old

        # --- Time the NEW Function ---
        start_new = time.perf_counter()
        new_results = []
        for s in strings:
            for mode in MODES:
                new_results.append(get_uni_rr(s, max=mode))
        end_new = time.perf_counter()
        time_new = end_new - start_new

        # --- Output Verification Check ---
        if old_results != new_results:
            print(f"\n❌ CRITICAL ERROR: Output mismatch detected at size {size}!")
            return

        # --- Calculate and Print Row Results ---
        speedup = time_old / time_new if time_new > 0 else float('inf')
        print(f"{size:<12} | {time_old:<14.6f} | {time_new:<14.6f} | {speedup:.2f}x faster")

    print("=" * 60)
    print("✅ All outputs matched exactly across all size categories.")

# Run the size-breakdown benchmark
benchmark_by_size_with_seed()

📊 Running size-isolated differential testing...
String Size  | Old Time (s)   | New Time (s)   | Speedup Factor
------------------------------------------------------------
4            | 0.000162       | 0.000133       | 1.21x faster
8            | 0.003863       | 0.002682       | 1.44x faster
16           | 0.727872       | 0.551297       | 1.32x faster
2048         | 0.073110       | 0.049560       | 1.48x faster
✅ All outputs matched exactly across all size categories.
